# HotpotQA — tokens, retrieval, and type slices (full 7,405)

bridge vs comparison on the **full distractor validation**, not a 24-Q toy slice.

Grounding (2025–26):
- **RAGAS / production eval:** split retrieval vs generation. We show title-overlap recall when gold docs exist, plus question/gold size by type.
- **CodeRAG-Bench (NAACL 2025):** retrieval is the bottleneck; nDCG/recall matter more than a single composite.
- **GraphRAG-Bench (ICLR 2026):** slice by task difficulty, never crown a winner from Hotpot averages.
- **Compact LLMs (SETN 2026):** 3B generators are enough when retrieval is strong; this harness scores locally with `llama3.2:3b`.

Rebuild after more scores land: re-run this notebook. Live Pages: https://jimmerz52-apple.github.io/rag-comparisons/


In [ ]:
from pathlib import Path
import sys
from IPython.display import Markdown, display
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from rag_benchmark.token_breakdown import load_bench_frame, plot_type_overview, BENCHES
payload = load_bench_frame(ROOT, 'hotpot')
spec = payload["spec"]
meta = payload["meta"]
display(Markdown(f"### {spec['title']}"))
display(Markdown(spec["types_help"]))
display(Markdown(
    f"Indexed **{meta.get('n_questions')}** Q / **{meta.get('n_documents')}** docs. "
    f"Scored rows in this slice: **{len(payload['frame'])}**. "
    f"Per-question LLM tokens: **{payload['has_llm_tokens']}**."
))
for n in payload["notes"]:
    display(Markdown(f"> {n}"))
display(Markdown("#### Catalog size by type (full indexed set)"))
display(payload["type_catalog"].round(1))


## Method × question type

Each row is one method on one type. `n` is how many scored questions of that type — if this is still a 12-Q leftover from an old run mixed with a live full-set method, the dashboard banner will say partial.


In [ ]:
by = payload["by_type"]
cols = [c for c in [
    "method_label", "question_type", "n",
    "mean_question_tokens", "tokens_per_query",
    "prompt_tokens_per_query", "completion_tokens_per_query",
    "mean_latency_s", "p95_latency_s", "mean_composite", "mean_judge",
] if c in by.columns]
display(by[cols].round(3) if len(by) else Markdown("_No scored accuracy yet._"))
if len(by):
    fig = plot_type_overview(payload)
    display(fig)
    plt.close(fig)


## Method-level LLM token ledger

`token_results.csv` is phase totals (query / evaluation / index). Index tokens are **not** per-question cost. Serving tok/q ≈ (query + eval) / n.


In [ ]:
tok = payload["tok_method"]
display(tok if len(tok) else Markdown("_No token_results.csv yet._"))
